# Evaluation Labels

This notebook keeps the standard evaluation separate and runs a targeted label-query evaluation on a selected S3DIS area.

For each room prefix in the selected area, it samples up to 3 rooms. For each sampled room, it samples 3 labels that are actually present in the room and evaluates cosine-similarity query behavior for those labels.

### 1. Setup Repo & Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

REPO_DIR = '/content/Deep_learning_project'
DRIVE_ROOT = '/content/drive/MyDrive/DL_Project'
EVAL_FEATURES_NAME = 's3dis_area4'
DRIVE_FEATURES_DIR = f'{DRIVE_ROOT}/features/{EVAL_FEATURES_NAME}'
LOCAL_FEATURES_ROOT = '/content/local_features'
LOCAL_FEATURES_DIR = f'{LOCAL_FEATURES_ROOT}/{EVAL_FEATURES_NAME}'
BRANCH = 'dev/eval-demo'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Gandata/Deep_learning_project.git {REPO_DIR}

In [ ]:
%cd {REPO_DIR}
!git restore configs/train_mlp_s3dis.yaml notebooks/pyproject.toml notebooks/evaluation_labels.ipynb src/evaluate_labels.py
!git fetch origin
!git checkout -B {BRANCH} origin/{BRANCH}
!git pull --no-edit origin {BRANCH}
!git branch --show-current
!git log -1 --oneline

### 2. Wire Drive Paths and Copy Features Locally

We keep `data/` and `checkpoints/` on Drive, but copy the selected feature folder to local runtime storage for faster evaluation.

In [ ]:
%cd {REPO_DIR}
!rm -f data checkpoints features pretrained
!ln -sf {DRIVE_ROOT}/data ./data
!ln -sf {DRIVE_ROOT}/checkpoints ./checkpoints
!mkdir -p {LOCAL_FEATURES_ROOT}
!rm -rf {LOCAL_FEATURES_DIR}
!cp -r {DRIVE_FEATURES_DIR} {LOCAL_FEATURES_ROOT}/
!ln -sf {LOCAL_FEATURES_ROOT} ./features
!echo Selected feature folder: {EVAL_FEATURES_NAME}
!readlink -f ./features/{EVAL_FEATURES_NAME}
!du -sh {LOCAL_FEATURES_DIR}
!ls -lh ./checkpoints/mlp_s3dis | tail

### 3. Setup Environment & Hugging Face Token

In [ ]:
%cd /content/Deep_learning_project/notebooks

from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
with open('.env', 'w') as f:
    f.write(f'HF_TOKEN={HF_TOKEN}\n')

!uv python install 3.10.12
!uv sync --python 3.10.12

### 4. Run Label-Query Evaluation

This keeps the standard full-feature evaluation untouched. Here we only use the selected area, sample up to 3 rooms per room prefix, then sample 3 labels present in each room and evaluate cosine-similarity query behavior.

In [ ]:
CONFIG_PATH = '/content/Deep_learning_project/configs/train_mlp_s3dis.yaml'
RESULTS_DIR = '/content/drive/MyDrive/DL_Project/results/evaluation/04b'
CHECKPOINT_NAME = 'last_model.pth'
CHECKPOINT_PATH = f'/content/Deep_learning_project/checkpoints/mlp_s3dis/{CHECKPOINT_NAME}'
EVAL_BATCH_SIZE = 16384
ROOMS_PER_PREFIX = 3
LABELS_PER_ROOM = 3
RANDOM_SEED = 42

print('Config:', CONFIG_PATH)
print('Results dir:', RESULTS_DIR)
print('Features dir:', f'/content/Deep_learning_project/features/{EVAL_FEATURES_NAME}')
print('Checkpoint:', CHECKPOINT_PATH)
print('Batch size:', EVAL_BATCH_SIZE)
print('Rooms per prefix:', ROOMS_PER_PREFIX)
print('Labels per room:', LABELS_PER_ROOM)
print('Seed:', RANDOM_SEED)

!PYTHONPATH=/content/Deep_learning_project uv run python /content/Deep_learning_project/src/evaluate_labels.py --config {CONFIG_PATH} --features_dir /content/Deep_learning_project/features/{EVAL_FEATURES_NAME} --checkpoint {CHECKPOINT_PATH} --batch_size {EVAL_BATCH_SIZE} --results_dir {RESULTS_DIR} --rooms_per_prefix {ROOMS_PER_PREFIX} --labels_per_room {LABELS_PER_ROOM} --seed {RANDOM_SEED}

### 5. Visualize the Saved Label-Query Results

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

AREA_JSON_PATH = Path(RESULTS_DIR) / 'area_level' / f'{EVAL_FEATURES_NAME}_{Path(CHECKPOINT_NAME).stem}_summary.json'
PREFIX_CSV_PATH = Path(RESULTS_DIR) / 'prefix_level' / f'{EVAL_FEATURES_NAME}_{Path(CHECKPOINT_NAME).stem}_prefix_metrics.csv'
ROOM_CSV_PATH = Path(RESULTS_DIR) / 'room_level' / f'{EVAL_FEATURES_NAME}_{Path(CHECKPOINT_NAME).stem}_room_metrics.csv'
QUERY_CSV_PATH = Path(RESULTS_DIR) / 'query_level' / f'{EVAL_FEATURES_NAME}_{Path(CHECKPOINT_NAME).stem}_query_metrics.csv'

print('Area summary file:', AREA_JSON_PATH)
print('Prefix metrics file:', PREFIX_CSV_PATH)
print('Room metrics file:', ROOM_CSV_PATH)
print('Query metrics file:', QUERY_CSV_PATH)

with AREA_JSON_PATH.open('r', encoding='utf-8') as handle:
    area_summary = json.load(handle)

prefix_df = pd.read_csv(PREFIX_CSV_PATH)
room_df = pd.read_csv(ROOM_CSV_PATH)
query_df = pd.read_csv(QUERY_CSV_PATH)

display(pd.Series(area_summary))
display(prefix_df[['room_prefix', 'num_selected_rooms', 'mean_average_precision', 'mean_topk_iou', 'mean_cosine_gap']].sort_values('mean_average_precision', ascending=False))
display(room_df[['room_file', 'room_prefix', 'mean_average_precision', 'mean_topk_iou', 'mean_cosine_gap']].sort_values('mean_average_precision', ascending=False).head(15))
display(query_df[['room_file', 'label_name', 'average_precision', 'roc_auc', 'topk_iou', 'positive_mean_cosine', 'negative_mean_cosine', 'cosine_gap']].sort_values('average_precision', ascending=False).head(20))

plt.figure(figsize=(12, 5))
plt.bar(prefix_df['room_prefix'], prefix_df['mean_average_precision'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('Mean AP')
plt.ylim(0.0, 1.0)
plt.title(f'Prefix-level mean AP - {EVAL_FEATURES_NAME} - {Path(CHECKPOINT_NAME).stem}')
plt.tight_layout()
plt.show()

room_plot_df = room_df.sort_values('mean_average_precision', ascending=False).reset_index(drop=True)
plt.figure(figsize=(14, 5))
plt.bar(room_plot_df['room_file'], room_plot_df['mean_average_precision'])
plt.xticks(rotation=90)
plt.ylabel('Mean AP')
plt.ylim(0.0, 1.0)
plt.title(f'Room-level mean AP - {EVAL_FEATURES_NAME} - {Path(CHECKPOINT_NAME).stem}')
plt.tight_layout()
plt.show()